## 04 -- Random Forest Regressor

Trains a Random Forest to predict next-24-hour mean PM2.5. Uses a
temporal train/test split (test = 2024-Q4) and RandomizedSearchCV with
TimeSeriesSplit cross-validation.

In [1]:
import pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report
warnings.filterwarnings('ignore')

In [2]:
# Load feature-engineered dataset
PROC = pathlib.Path('../data/processed')
df = pd.read_csv(PROC / 'features_engineered.csv', parse_dates=['datetime'])

FEATURE_COLS = [
    'o3', 'no2', 'pm25',
    'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
    'wind_direction_10m', 'precipitation', 'surface_pressure',
    'hour', 'day_of_week', 'month', 'is_weekend',
    'pm25_lag_1', 'pm25_lag_2', 'pm25_lag_3', 'pm25_lag_24',
    'pm25_roll_24h', 'pm25_roll_72h',
]
TARGET_COL = 'pm25_next24h'

# Drop rows with any NaN in features or target
model_df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()
print(f'Rows available: {len(model_df):,}  (dropped {len(df) - len(model_df):,} with NaN)')
print(f'Date range    : {model_df["datetime"].min().date()}  to  {model_df["datetime"].max().date()}')
model_df.head(3)

Rows available: 74,543  (dropped 6,485 with NaN)
Date range    : 2023-01-02  to  2024-12-30


,datetime,city,o3,no2,pm25,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation,...,day_of_week,month,is_weekend,pm25_lag_1,pm25_lag_2,pm25_lag_3,pm25_lag_24,pm25_roll_24h,pm25_roll_72h,pm25_next24h
34,2023-01-02 13:00:00,Birmingham A4540 Roadside,35.62324,32.43474,3.797,6.2,79,9.7,243,0.0,...,0,1,0,3.986,4.198,4.104,8.632,5.766542,5.710833,2.995
35,2023-01-02 14:00:00,Birmingham A4540 Roadside,33.27830,42.81079,4.080,6.2,79,11.4,242,0.0,...,0,1,0,3.797,3.986,4.198,9.033,5.565083,5.659108,3.042
36,2023-01-02 15:00:00,Birmingham A4540 Roadside,32.57980,41.95037,4.599,5.5,85,9.4,238,0.0,...,0,1,0,4.080,3.797,3.986,9.646,5.358708,5.617553,4.646


In [3]:
# Temporal train / test split  -- test = 2024-Q4 (Oct-Dec)
SPLIT_DATE = '2024-10-01'

train = model_df[model_df['datetime'] < SPLIT_DATE]
test  = model_df[model_df['datetime'] >= SPLIT_DATE]

X_train, y_train = train[FEATURE_COLS].values, train[TARGET_COL].values
X_test,  y_test  = test[FEATURE_COLS].values,  test[TARGET_COL].values

print(f'Train: {len(train):,} rows   ({train["datetime"].min().date()} to {train["datetime"].max().date()})')
print(f'Test : {len(test):,} rows    ({test["datetime"].min().date()} to {test["datetime"].max().date()})')

Train: 64,049 rows   (2023-01-02 to 2024-09-30)
Test : 10,494 rows    (2024-10-01 to 2024-12-30)


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# 2x2 grid = 8 fits total (4 combos x 2-fold) -- keeps runtime < 2 min
param_grid = {
    'n_estimators': [100, 200],
    'max_depth':    [15, None],
}
tscv = TimeSeriesSplit(n_splits=2)

search = GridSearchCV(
    RandomForestRegressor(min_samples_leaf=2, n_jobs=2, random_state=42),
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=1,
    verbose=1,
)
search.fit(X_train, y_train)

print('Best params:', search.best_params_)
print(f'Best CV RMSE: {-search.best_score_:.3f} ug/m3')

Fitting 2 folds for each of 4 candidates, totalling 8 fits


Best params: {'max_depth': None, 'n_estimators': 200}
Best CV RMSE: 4.808 ug/m3


In [5]:
from sklearn.ensemble import RandomForestRegressor
import joblib

best_rf = RandomForestRegressor(**search.best_params_, n_jobs=-1, random_state=42)
best_rf.fit(X_train, y_train)
y_pred = best_rf.predict(X_test)

In [6]:
# DAQI PM2.5 band classification (24-h mean, ug/m3)
def daqi_band(x):
    if x < 12:  return 'Low'
    elif x < 24: return 'Moderate'
    elif x < 48: return 'High'
    else:        return 'Very High'

y_pred = np.clip(y_pred, 0, None)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
print(f'Test RMSE : {rmse:.3f} ug/m3')
print(f'Test MAE  : {mae:.3f} ug/m3')

y_true_band = [daqi_band(v) for v in y_test]
y_pred_band = [daqi_band(v) for v in y_pred]
bands = ['Low', 'Moderate', 'High', 'Very High']
print()
print('DAQI band classification:')
print(classification_report(y_true_band, y_pred_band, labels=bands, zero_division=0))

Test RMSE : 5.252 ug/m3
Test MAE  : 3.918 ug/m3

DAQI band classification:
              precision    recall  f1-score   support

         Low       0.86      0.92      0.89      8353
    Moderate       0.38      0.31      0.34      1782
        High       0.67      0.03      0.05       356
   Very High       0.00      0.00      0.00         3

    accuracy                           0.79     10494
   macro avg       0.48      0.32      0.32     10494
weighted avg       0.77      0.79      0.77     10494



In [7]:
MODEL_DIR = pathlib.Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

importances = pd.Series(best_rf.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Feature importance')
ax.set_title('Random Forest -- top 15 features')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'rf_feature_importance.png', dpi=120)
plt.show()
print('Feature importance plot saved.')

Feature importance plot saved.


In [8]:
import joblib
MODEL_DIR = pathlib.Path('../models')
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(best_rf, MODEL_DIR / 'random_forest.pkl')
print('Model saved to models/random_forest.pkl')

Model saved to models/random_forest.pkl
